# MACHINE LEARNING MODELS

## IMPORTS

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os

##  DataLoader Class

In [3]:
class DiabeticDataLoader:
    def __init__(self, df_raw):
        self.df_raw = df_raw.copy()
        self.df_clean = None
        self.df_no_outliers = None
        self.scaler = None

    def clean_data(self):
        df = self.df_raw.copy()

        # Remover colunas constantes
        constant_cols = ['examide', 'citoglipton']
        df.drop(columns=constant_cols, inplace=True, errors='ignore')

        # Substituir '?' por NaN de todas as colunas
        df.replace('?', np.nan, inplace=True)

        # Lidar com max_glu_serum e A1Cresult
        for col in ['max_glu_serum', 'A1Cresult']:
            df[col] = df[col].replace('None', np.nan)
            df[col + '_measured'] = (~df[col].isna()).astype(int)

        # Indicador de peso registrado e conversão
        df['weight_recorded'] = (~df['weight'].isna()).astype(int)
        df['weight'] = df['weight'].astype('category')

        # Colunas categóricas
        categorical_cols = [
            'race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id',
            'admission_source_id', 'payer_code', 'medical_specialty',
            'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult',
            'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
            'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
            'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
            'insulin', 'glyburide-metformin', 'glipizide-metformin',
            'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone',
            'change', 'diabetesMed', 'readmitted'
        ]

        for col in categorical_cols:
            if col in df.columns:
                df[col] = df[col].astype('category')

        self.df_clean = df
        return df

    def remove_outliers(self, df=None):
        if df is None:
            if self.df_clean is None:
                raise ValueError("Data must be cleaned before outlier removal.")
            else:
                df = self.df_clean.copy()

        outlier_cols = [
            'time_in_hospital', 'num_lab_procedures', 'num_medications',
            'number_outpatient', 'number_emergency', 'number_inpatient'
        ]

        self.thresholds = {}
        for col in outlier_cols:
            if col in df.columns:
                # Calcula Min e P99 para ser o thresholds
                low = df[col].min()
                high = df[col].quantile(0.99)
                self.thresholds[col] = (low, high)

        mask = pd.Series(True, index=df.index)
        for feature, (low, high) in self.thresholds.items():
            if feature in df.columns:
                mask &= df[feature].between(low, high)

        df_no_outliers = df.loc[mask].copy()
        self.df_no_outliers = df_no_outliers
        return df_no_outliers

    def get_clean_data(self):
        if self.df_clean is None:
            self.clean_data()
        return self.df_clean

    def get_no_outliers_data(self):
        if self.df_no_outliers is None:
            self.remove_outliers()
        return self.df_no_outliers

    def get_features_target(self):
        df = self.get_no_outliers_data()
        exclude_cols = ['encounter_id', 'patient_nbr', 'readmitted']
        feature_cols = [col for col in df.columns if col not in exclude_cols]
        X = df[feature_cols]
        y = df['readmitted']
        return X, y

    def get_train_test_split(self, test_size=0.2, random_state=42):
        # Limpa os dados de forma global
        df_clean = self.clean_data().dropna()

        # Separa X e Y
        X, y = self.get_features_target()

        # Faz o split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, stratify=y, random_state=random_state
        )

        # Remove outliers no train
        train_combined = pd.concat([X_train, y_train], axis=1)
        train_combined = self.remove_outliers(train_combined)
        X_train = train_combined.drop(columns=['readmitted'])
        y_train = train_combined['readmitted']

        # Padronização
        numeric_cols = [
            'time_in_hospital', 'num_lab_procedures', 'num_medications',
            'number_outpatient', 'number_emergency', 'number_inpatient',
            'num_procedures', 'number_diagnoses'
        ]

        self.scaler = StandardScaler()
        # FIT apenas no treino
        X_train[numeric_cols] = self.scaler.fit_transform(X_train[numeric_cols])
        # TRANSFORM no teste
        X_test[numeric_cols] = self.scaler.transform(X_test[numeric_cols])

        return X_train, X_test, y_train, y_test

### How to Use It

| # | Method                       | Description                                |
|---|------------------------------|--------------------------------------------|
| 1 | `get_clean_data()`           | Full cleaned dataframe                     |
| 1 | `get_outlier_treated_data()` | Flagging  outliers                         |
| 1 | `get_no_outliers_data()`     | Removing outliers                          |
| 1 | `standardize_features()`     | Standardizing features                     |
| 2 | `get_features_target()`      | Modeling features and target (no outliers) |
| 3 | `get_train_test_split()`     | Reproducible encoded train/test split      |

# INITIAL ANALYSIS

## IMPORTS

In [ ]:
# %pip install lightgbm xgboost catboost

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, TargetEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score
import sklearn.metrics as metrics

## SETTING UP DATA

In [ ]:
df_diabetic_data = pd.read_csv(os.path.join('..','data', 'raw','diabetic_data.csv')) 
# df_diabetic_data = pd.read_csv('/content/diabetic_data.csv') # Pra usar no google colab
df_diabetic_loader = DiabeticDataLoader(df_diabetic_data)

df_clean = df_diabetic_loader.clean_data()
df_clean = df_clean.drop(columns=['encounter_id', 'patient_nbr']) # Colunas que não importam para o modelo

df_clean.columns

Index(['race', 'gender', 'age', 'weight', 'admission_type_id',
       'discharge_disposition_id', 'admission_source_id', 'time_in_hospital',
       'payer_code', 'medical_specialty', 'num_lab_procedures',
       'num_procedures', 'num_medications', 'number_outpatient',
       'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3',
       'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin',
       'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
       'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted',
       'max_glu_serum_measured', 'A1Cresult_measured', 'weight_recorded'],
      dtype='object')

In [7]:
X_train, X_test, y_train, y_test = df_diabetic_loader.get_train_test_split()

## TESTING MODELS

In [ ]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

ordinal_cols = [
    'age', 'weight'
]

target_cols = [
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
    'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3'
]

numerical_cols = [
    'time_in_hospital', 'num_lab_procedures', 'num_medications',
    'number_outpatient', 'number_emergency', 'number_inpatient',
    'num_procedures', 'number_diagnoses'
]

dummy_cols = [
    'troglitazone', 'race', 'gender', 'glimepiride', 'acetohexamide',
    'metformin-pioglitazone', 'tolazamide', 'max_glu_serum_measured',
    'metformin-rosiglitazone', 'glipizide-metformin', 'pioglitazone',
    'change', 'max_glu_serum', 'glyburide-metformin', 'glipizide',
    'nateglinide', 'miglitol', 'metformin', 'weight_recorded',
    'rosiglitazone', 'chlorpropamide', 'A1Cresult', 'insulin',
    'A1Cresult_measured', 'tolbutamide', 'acarbose', 'repaglinide',
    'glimepiride-pioglitazone', 'diabetesMed', 'glyburide'
]

# Pipeline de pré-processamento (primeiro faz o imputation, depois o encoding dependendo do tipo de coluna)
preprocessor = ColumnTransformer(
    transformers=[
        # Colunas categóricas com poucas categorias
        ('dummy', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), dummy_cols),

        # Colunas categóricas com muitas categorias ordinais
        ('ordinal', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
        ]), ordinal_cols),

        # Colunas categóricas com muitas categorias não-ordinais
        ('target', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('encoder', TargetEncoder())
        ]), target_cols),

        # Colunas numéricas
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numerical_cols)
    ],
    remainder='drop'
).set_output(transform="pandas") # Retorna um df do pandas em vez de um array numpy (alguns modelos precisam dos nomes das features)

# Lista dos modelos a serem testados na primeira "iteração" (separei em duas iterações pra execução não demorar muito + facilidade de testar)
models_first_iteration = {
    "SGD": SGDClassifier(max_iter=2000, class_weight='balanced'),
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight='balanced'),
    "KNN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', max_depth=10),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight='balanced'),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, class_weight='balanced'),
}

# Testa e imprime o resultado de cada modelo
for name, model in models_first_iteration.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    # Cross-validation 
    cv_scores = cross_val_score(pipeline, X_train, y_train_encoded, cv=5)

    pipeline.fit(X_train, y_train_encoded)
    y_pred = pipeline.predict(X_test)

    print(f"\n{'-'*30}")
    print(f"Modelo: {name}")
    print(f"Acurácia média do Cross Evaluation: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
    print(f"Acurácia da Predição: {metrics.accuracy_score(y_test_encoded, y_pred):.4f}")
    print(f"\n\n{metrics.classification_report(y_test_encoded, y_pred, target_names=le.classes_)}")


------------------------------
Modelo: SGD
Acurácia média do Cross Evaluation: 0.5532 (+/- 0.0988)
Acurácia da Predição: 0.5718


              precision    recall  f1-score   support

         <30       0.25      0.08      0.12      2111
         >30       0.52      0.23      0.32      6767
          NO       0.60      0.89      0.71     10643

    accuracy                           0.57     19521
   macro avg       0.46      0.40      0.38     19521
weighted avg       0.53      0.57      0.51     19521


------------------------------
Modelo: Logistic Regression
Acurácia média do Cross Evaluation: 0.5013 (+/- 0.0112)
Acurácia da Predição: 0.4965


              precision    recall  f1-score   support

         <30       0.19      0.41      0.26      2111
         >30       0.45      0.39      0.42      6767
          NO       0.69      0.58      0.63     10643

    accuracy                           0.50     19521
   macro avg       0.44      0.46      0.44     19521
weighted avg   

In [ ]:
models_second_iteration = {
    "LightGBM": LGBMClassifier(class_weight='balanced', random_state=42),
    "XGBoost": XGBClassifier(eval_metric='mlogloss', random_state=42),
    "CatBoost": CatBoostClassifier(verbose=0, auto_class_weights='Balanced', random_state=42),
    "LDA": LinearDiscriminantAnalysis()
}

for name, model in models_second_iteration.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])


    # Cross-validation
    cv_scores = cross_val_score(pipeline, X_train, y_train_encoded, cv=5)

    pipeline.fit(X_train, y_train_encoded)
    y_pred = pipeline.predict(X_test)

    print(f"\n{'-'*30}")
    print(f"Modelo: {name}")
    print(f"Acurácia média do Cross Evaluation: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
    print(f"Acurácia da Predição: {metrics.accuracy_score(y_test_encoded, y_pred):.4f}")
    print(f"\n\n{metrics.classification_report(y_test_encoded, y_pred, target_names=le.classes_)}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.058645 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4062
[LightGBM] [Info] Number of data points in the train set: 59964, number of used features: 103
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.038384 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4054
[LightGBM] [Info] Number of data points in the train set: 59964, number of used features: 103
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Star